# A* Pathfinding on Weighted graphs with Code and Experiments

**AM6020 Written report: Supporting notebook**

Authors: 

Coarse: Mathematical Modelling and Machine Learning

Date: 14th November 2026

## About this notebook

In this notebook we are goind to cover the code behind the written report *A Study and implementation of A\* Pathfinding Algorithm. The report is written in latex (overleaf). 
In this notebook we have implmented the algorithms, checks each algorithm written from scratch against the `networks` library, runs the experiments, and saves the tables and figures for the report.

We have written the search algorithms from scratch using numpy and python's standard library. We have used `Networkx` to check if our algorithms results are aligned with the results generated by using this pre built libraries and we have also used this to build the graphs. `osmnx` is used to prepare the Cork road network. 

**Setup:** This section loads all the libraries required, fixes the random seed so the results can be reproduced, creates a folder for figures and tables, sets plotting style so the size of each figure looks same. 

In [1]:
import time
import math
import random
import heapq # binary heap used for A* open list. 
import platform
from collections import deque # FIFO queue used for BFS
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib
import osmnx as ox # this is only used to download the Cork road network once. 

In [2]:
#Output folder:
FIG_DIR = Path("figures")
TABLE_DIR = Path("tables")
DATA_DIR = Path("data")
for d in [FIG_DIR, TABLE_DIR, DATA_DIR]:
    d.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (6, 4),
    "figure.dpi": 110,
    "font.size": 11,
    "font.family": "serif",          # matches the LaTeX body text
    "axes.titlesize": 11,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,              # embeds fonts properly in the PDF
})

In [3]:
def save_fig(fig, name):
    path = FIG_DIR / f"{name}.pdf"
    fig.savefig(path)
    print(f"Saved {path}")

def save_table(df, name, caption=None, label=None, float_format="%.2f"):
    path = TABLE_DIR / f"{name}.tex"
    latex = df.to_latex(index=False, float_format=float_format,
                        caption=caption, label=label, position="htbp")
    path.write_text(latex, encoding="utf-8")
    print(f"Saved {path}")

versions = pd.DataFrame({
    "Package": ["Python", "NumPy", "pandas", "NetworkX", "Matplotlib", "OSMnx"],
    "Version": [platform.python_version(), np.__version__, pd.__version__,
                nx.__version__, matplotlib.__version__,
                ox.__version__],
})
print(versions.to_string(index=False))
save_table(versions, "versions", caption="Software versions used.",
           label="tab:versions", float_format=None)

   Package Version
    Python  3.11.0
     NumPy   2.4.6
    pandas   2.2.3
  NetworkX   3.6.1
Matplotlib  3.10.1
     OSMnx   2.1.1
Saved tables\versions.tex


## 2. Implementation

This section implements the search algorithms from scratch. The only tools
used are Python's standard library (`heapq` for the priority queue, `deque`
for the BFS queue) and NumPy. NetworkX supplies the graph data structure
(`G[u]` gives the neighbours of node `u` and the edge weights), but none of
its search functions are used here. Those are used only in Section 3, to
check our results.

The design follows the best-first search framework of Russell & Norvig
(2020, Ch. 3). A single function expands nodes in order of an evaluation
function f(n), and the choice of f gives each algorithm:

| Algorithm | Evaluation function |
|---|---|
| Dijkstra | f(n) = g(n) |
| A\* | f(n) = g(n) + h(n) |
| Weighted A\* | f(n) = g(n) + w·h(n), with w ≥ 1 |
| Greedy best-first | f(n) = h(n) |

Breadth-first search is written as a separate function, since it uses a
FIFO queue and ignores edge weights.

### 2.1 Result container and helper functions

Every search returns the same set of results, so the algorithms can be
compared directly in later sections. Along with the path and its cost, we
record the number of nodes expanded, the number generated (pushed onto the
open list) and the peak size of the open list. The peak open-list size is
our measure of memory use.

In [ ]:
from dataclasses import dataclass, field
@dataclass
class SearchResult:
    path: list
    cost: float
    expanded: int
    generated: int
    peak_open: int
    order: list = field(default_factory=list) # what this means is it creates a python list everytime the object is called. 



In [1]:
# understand the dijkstra algorithm and how it works.
graph = {
    'A': {'B': 4, 'D': 7},
    'B': {'A': 4, 'C': 2, 'D': 5},
    'C': {'B': 2, 'D': 3},
    'D': {'A': 7, 'B': 5, 'C': 3}
}

In [4]:
distance = {}
for node in graph:
    distance[node] = float('inf')
    
distance['A'] = 0

